# Day 9 — Silver: Reports (Bronze JSON → Silver Delta, Single Report Type)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/reports/YYYY/MM/`

**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/reports/` (Delta)

## Report types in Bronze

Each month Bronze stores three JSON files:
- `kpi_report_YYYYMM.json`    — KPI metrics (energy, cost, utilisation)
- `sla_report_YYYYMM.json`    — SLA metrics (uptime, response time)
- `state_breakdown_YYYYMM.json` — Charger state distribution

## What Silver does

This v1 notebook processes **kpi_report only** to demonstrate the pattern:
1. Read monthly JSON file with `spark.read.json()` (multiLine)
2. Flatten nested fields using `select()` with dot-notation
3. Add type-safe columns + audit metadata
4. Write to Silver Delta partitioned by `report_year` and `report_month`

v2 adds all three report types via a for loop.
v3 is production with job parameters.

## Job parameters (for v3 reference)
```
load_type       : incremental
ingestion_year  : 2026
ingestion_month : 06
```

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Parameters (hardcoded for learning — v3 uses job params)
LOAD_TYPE       = "incremental"
INGESTION_YEAR  = "2026"
INGESTION_MONTH = "06"

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")

In [ ]:
# Cell 3 — Constants
BRONZE_REPORTS = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/reports"
SILVER_REPORTS = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/reports"

YYYYMM = f"{INGESTION_YEAR}{INGESTION_MONTH}"

# v1 processes kpi_report only
KPI_FILE   = f"{BRONZE_REPORTS}/{INGESTION_YEAR}/{INGESTION_MONTH}/kpi_report_{YYYYMM}.json"
SILVER_KPI = f"{SILVER_REPORTS}/kpi_report"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze KPI file : {KPI_FILE}")
print(f"Silver KPI path : {SILVER_KPI}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Read Bronze KPI report JSON
# Use multiLine=true because report JSON is formatted across multiple lines.
# recursiveFileLookup=true reads all JSON files in sub-paths if any.

kpi_raw = (
    spark.read
    .option("multiLine", True)
    .json(KPI_FILE)
)

print("Bronze KPI schema (raw):")
kpi_raw.printSchema()
kpi_raw.show(3, truncate=False)

In [ ]:
# Cell 5 — Explore nested structure
# The KPI report is a monthly summary document.
# Typical structure:
#   {
#     "report_period": "2026-06",
#     "generated_at": "2026-07-01T00:00:00Z",
#     "summary": {
#       "total_sessions": 18420,
#       "total_energy_kwh": 924500.0,
#       "total_revenue_aud": 461850.0,
#       "avg_session_duration_min": 48.3,
#       "avg_energy_per_session_kwh": 50.2
#     },
#     "by_charger_type": [
#       { "charger_type": "DC_FAST", "sessions": 9210, "energy_kwh": 640000 },
#       { "charger_type": "AC_SLOW", "sessions": 9210, "energy_kwh": 284500 }
#     ],
#     "by_state": [
#       { "state": "NSW", "sessions": 5526, "revenue_aud": 138555 },
#       ...
#     ]
#   }

print("Top-level columns:")
print(kpi_raw.columns)
print()
print("Full schema:")
kpi_raw.printSchema()

In [ ]:
# Cell 6 — Flatten KPI summary (top-level fields + summary sub-object)
# Silver stores one row per report_period (month).
# Nested array fields (by_charger_type, by_state) go to separate Silver tables in v2+.

kpi_flat = kpi_raw.select(
    F.col("report_period").cast("string").alias("report_period"),
    F.to_timestamp(F.col("generated_at")).alias("generated_at"),
    # Summary sub-object fields
    F.col("summary.total_sessions").cast("long").alias("total_sessions"),
    F.col("summary.total_energy_kwh").cast("decimal(14,2)").alias("total_energy_kwh"),
    F.col("summary.total_revenue_aud").cast("decimal(14,2)").alias("total_revenue_aud"),
    F.col("summary.avg_session_duration_min").cast("decimal(8,2)").alias("avg_session_duration_min"),
    F.col("summary.avg_energy_per_session_kwh").cast("decimal(8,2)").alias("avg_energy_per_session_kwh"),
    # Partition helpers
    F.lit(int(INGESTION_YEAR)).alias("report_year"),
    F.lit(int(INGESTION_MONTH)).alias("report_month"),
    # Audit
    F.lit(RUN_TS).cast("timestamp").alias("silver_ingested_at"),
    F.lit(LOAD_TYPE).alias("silver_load_type"),
    F.lit("pl_silver_reports_v1").alias("silver_pipeline"),
)

print("Flattened KPI schema:")
kpi_flat.printSchema()
kpi_flat.show(3, truncate=False)

In [ ]:
# Cell 7 — Write KPI Silver (overwrite for v1 — MERGE in v3)

(
    kpi_flat.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_KPI)
)

print(f"Wrote {kpi_flat.count()} rows to {SILVER_KPI}")

In [ ]:
# Cell 8 — Verify

verify_df = spark.read.format("delta").load(SILVER_KPI)
print(f"Silver KPI rows: {verify_df.count()}")
verify_df.show(truncate=False)
verify_df.printSchema()